# 11 — Hands-On Lab
## Week 1 Capstone: NumPy Linear Algebra

> **Prerequisites:** All of Notebooks 00–10  
> **Time:** ~2–3 hours  
> **Goal:** Implement key concepts from scratch and verify them against library functions

---

## What you'll build

| Part | Task | Concepts used |
|---|---|---|
| **1** | Matrix multiplication from scratch | Notebook 03: dot products |
| **2** | Eigendecomposition + verification | Notebook 05: eigenvalues |
| **3** | PCA by hand using SVD | Notebook 06: SVD |
| **4** | Visualize column space & null space | Notebook 07: rank/null space |
| **5** | Compare numpy.linalg.eig vs scipy.linalg.eig | Notebooks 05, 10 |

---

> **How to use this lab:**
> 1. Read the explanation in each section
> 2. Try to understand the code BEFORE running it
> 3. Run each cell and verify the outputs make sense
> 4. Try the **Challenge** questions at the end of each section

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as scipy_linalg
import time

np.random.seed(42)
print("All libraries loaded. Let's build things!")

---
# Part 1: Matrix Multiplication From Scratch

## The goal
Implement matrix multiplication using only Python loops — no `@`, no `np.dot`, no `np.matmul`.

This forces you to understand **exactly what's happening** in every matrix multiply your model performs.

## Recall the formula
For C = A × B where A is (m×k) and B is (k×n):
> C[i, j] = Σₗ A[i, l] × B[l, j]

That's a dot product of row i from A with column j from B.

In [ ]:
# ─────────────────────────────────────────────────────────
# YOUR IMPLEMENTATION
# ─────────────────────────────────────────────────────────

def matmul_scratch(A, B):
    """
    Matrix multiply A @ B using only Python loops.
    
    A: numpy array of shape (m, k)
    B: numpy array of shape (k, n)
    Returns: C of shape (m, n)
    """
    m, k = A.shape        # A has m rows and k columns
    k2, n = B.shape       # B has k2 rows and n columns
    
    # Check that inner dimensions match
    if k != k2:
        raise ValueError(f"Incompatible shapes: A={A.shape}, B={B.shape}")
    
    # Initialize result with zeros
    C = np.zeros((m, n))
    
    # Triple loop: for each output cell C[i,j]...
    for i in range(m):       # loop over rows of A
        for j in range(n):   # loop over columns of B
            # ...compute the dot product of row i of A with column j of B
            for l in range(k):
                C[i, j] += A[i, l] * B[l, j]
    
    return C

# ── Test on a simple example ─────────────────────────────
A = np.array([[1., 2., 3.],
              [4., 5., 6.]])
B = np.array([[7.,  8.],
              [9.,  10.],
              [11., 12.]])

C_scratch = matmul_scratch(A, B)
C_numpy   = A @ B

print("=== Part 1: Matrix Multiply From Scratch ===")
print(f"A ({A.shape}):")
print(A)
print(f"\nB ({B.shape}):")
print(B)
print(f"\nC = A @ B using loops:")
print(C_scratch)
print(f"\nC = A @ B using NumPy:")
print(C_numpy)
print(f"\nResults match: {np.allclose(C_scratch, C_numpy)} ✓")

In [ ]:
# ── Step-by-step explanation ─────────────────────────────
print("=== Step-by-step: computing C[0,0] ===")
print()
print(f"Row 0 of A: {A[0]}")
print(f"Col 0 of B: {B[:,0]}")
print(f"Dot product: {A[0,0]}×{B[0,0]} + {A[0,1]}×{B[1,0]} + {A[0,2]}×{B[2,0]}")
print(f"           = {A[0,0]*B[0,0]} + {A[0,1]*B[1,0]} + {A[0,2]*B[2,0]}")
print(f"           = {A[0,0]*B[0,0] + A[0,1]*B[1,0] + A[0,2]*B[2,0]}")
print(f"C[0,0] = {C_scratch[0,0]}")  

In [ ]:
# ── Speed benchmark ──────────────────────────────────────
print("=== Speed comparison ===")

# Loops on a 25×25 matrix
size = 25
A_bench = np.random.randn(size, size)
B_bench = np.random.randn(size, size)

start = time.time()
C_loop = matmul_scratch(A_bench, B_bench)
time_loop = time.time() - start

# NumPy on a 500×500 matrix
size2 = 500
A2 = np.random.randn(size2, size2)
B2 = np.random.randn(size2, size2)
start = time.time()
for _ in range(50): C_np = A2 @ B2
time_np = (time.time() - start) / 50

print(f"Python loops ({size}×{size}):   {time_loop*1000:.1f} ms")
print(f"NumPy @      ({size2}×{size2}): {time_np*1000:.3f} ms")
print()
print("NumPy calls BLAS (Basic Linear Algebra Subprograms) — written in C/Fortran")
print("with SIMD (parallel) CPU instructions. That's why it's hundreds of times faster.")
print("For ML at scale, you MUST use @ or np.matmul — never write loops.")

---
# Part 2: Eigendecomposition with numpy.linalg

## The goal
1. Compute the eigendecomposition of a symmetric matrix
2. Verify: A = V Λ Vᵀ (reconstruction)
3. Verify: Av = λv for each pair (the defining equation)
4. Compute A^10 efficiently using eigenvalues

In [ ]:
# ─────────────────────────────────────────────────────────
# EIGENDECOMPOSITION
# ─────────────────────────────────────────────────────────

print("=== Part 2: Eigendecomposition ===")
print()

# Create a symmetric positive definite matrix (like a covariance matrix)
M = np.random.randn(5, 5)
A = M.T @ M + 3 * np.eye(5)   # PD: AᵀA + 3I guarantees all eigenvalues ≥ 3

print(f"Symmetric PD matrix A (5×5):")
print(A.round(3))
print()

# For symmetric matrices: use eigh (not eig)
# eigh is:  faster, more numerically stable, guarantees real eigenvalues
eigenvalues, V = np.linalg.eigh(A)   # eigenvalues sorted ascending
Lambda = np.diag(eigenvalues)        # diagonal matrix of eigenvalues

print(f"Eigenvalues (ascending): {eigenvalues.round(4)}")
print(f"All positive (PD confirmed): {(eigenvalues > 0).all()}")
print()
print(f"Eigenvector matrix V (columns = eigenvectors):")
print(V.round(4))

In [ ]:
# ── Verification 1: A = V Λ Vᵀ ──────────────────────────
A_reconstructed = V @ Lambda @ V.T
print("=== Verification 1: A = VΛVᵀ ===")
print(f"Reconstruction error: {np.linalg.norm(A - A_reconstructed):.2e}")
print(f"(Should be near 0 — close to machine precision)")
print()

# ── Verification 2: Av = λv ─────────────────────────────
print("=== Verification 2: Av = λv for each pair ===")
for i in range(len(eigenvalues)):
    v = V[:, i]           # i-th eigenvector (column i)
    lam = eigenvalues[i]  # i-th eigenvalue
    Av  = A @ v           # apply matrix to eigenvector
    lv  = lam * v         # scale eigenvector by eigenvalue
    error = np.max(np.abs(Av - lv))
    print(f"  λ{i+1} = {lam:.4f}: Av = λv? {np.allclose(Av, lv)} (max error: {error:.2e})")

print()
# ── Verification 3: Eigenvectors are orthonormal ─────────
print("=== Verification 3: VᵀV = I (orthonormal eigenvectors) ===")
VtV = V.T @ V
print(f"VᵀV is identity? {np.allclose(VtV, np.eye(5))}")
print(f"Max off-diagonal: {np.max(np.abs(VtV - np.eye(5))):.2e}")

In [ ]:
# ── Matrix power via eigendecomposition ─────────────────
print("=== Efficient matrix powers via eigendecomposition ===")
print("A^k = V @ diag(λ^k) @ Vᵀ  (just raise eigenvalues to the power k)")
print()

k = 10
# Via eigendecomposition (fast)
A10_eigen  = V @ np.diag(eigenvalues**k) @ V.T
# Via direct matrix multiplication (for comparison)
A10_direct = np.linalg.matrix_power(A, k)

print(f"A^{k} via eigendecomposition and via direct multiply:")
print(f"First 2×2 corner (eigen):  \n{A10_eigen[:2,:2].round(2)}")
print(f"First 2×2 corner (direct): \n{A10_direct[:2,:2].round(2)}")
print(f"Match: {np.allclose(A10_eigen, A10_direct, rtol=1e-5)}")
print()
print("Why is eigendecomposition better for large k?")
print("  Direct: requires k matrix multiplications")
print("  Eigen:  eigenvalues**k is trivial — just one power per eigenvalue!")

---
# Part 3: PCA by Hand Using SVD

## The goal
Implement PCA from scratch using SVD on a synthetic 2D dataset:
1. Generate correlated data
2. Center it
3. Apply SVD
4. Extract principal components
5. Project data
6. Visualize

In [ ]:
# ─────────────────────────────────────────────────────────
# PCA FROM SCRATCH USING SVD
# ─────────────────────────────────────────────────────────

print("=== Part 3: PCA by hand using SVD ===")
print()

# STEP 1: Generate a 2D dataset with clear structure
np.random.seed(42)
n = 300
t = np.random.randn(n)                # hidden underlying factor
x1 = 2*t + 0.4*np.random.randn(n)    # feature 1: strongly correlated with t
x2 = t + 0.4*np.random.randn(n)      # feature 2: correlated but noisier
X = np.column_stack([x1, x2])         # dataset: 300 × 2

print(f"Step 1: Dataset shape {X.shape}, correlation between x1,x2 = {np.corrcoef(x1,x2)[0,1]:.3f}")

# STEP 2: Center the data (REQUIRED for PCA — subtract the mean)
X_mean = X.mean(axis=0)    # mean of each column (feature)
X_c = X - X_mean           # center: make mean = 0
print(f"Step 2: Centered. Mean before: {X.mean(axis=0).round(3)}, after: {X_c.mean(axis=0).round(6)}")

# STEP 3: SVD of the centered data
U, s, Vt = np.linalg.svd(X_c, full_matrices=False)
V = Vt.T   # columns of V are principal component directions

print(f"Step 3: SVD done")
print(f"  U shape: {U.shape}")
print(f"  s (singular values): {s.round(3)}")
print(f"  Vt shape: {Vt.shape}")

# STEP 4: Principal components
print(f"\nStep 4: Principal component directions")
print(f"  PC1 = {V[:,0].round(4)}  (direction of most variance)")
print(f"  PC2 = {V[:,1].round(4)}  (perpendicular to PC1)")
print(f"  PC1 · PC2 = {np.dot(V[:,0], V[:,1]):.6e}  (orthogonal ✓)")

# STEP 5: Variance explained
variance_per_pc = s**2 / (n - 1)
total_var = variance_per_pc.sum()
pct_var = variance_per_pc / total_var * 100

print(f"\nStep 5: Variance explained")
print(f"  PC1: {pct_var[0]:.1f}% of total variance")
print(f"  PC2: {pct_var[1]:.1f}% of total variance")

# STEP 6: Project onto PCs (get scores)
X_pca = X_c @ V    # shape (300, 2) — data in PCA coordinate system
print(f"\nStep 6: Projected data shape: {X_pca.shape}")

In [ ]:
# ── Visualize PCA ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: original data with PC directions
ax = axes[0]
ax.scatter(X[:,0], X[:,1], alpha=0.3, s=15, color='steelblue')
for i, color in enumerate(['red', 'orange']):
    scale = np.sqrt(variance_per_pc[i]) * 2
    direction = V[:, i] * scale
    ax.annotate('', xy=X_mean + direction, xytext=X_mean,
                arrowprops=dict(arrowstyle='->', color=color, lw=3))
    ax.text(*(X_mean + direction * 1.2), f'PC{i+1}\n({pct_var[i]:.0f}%)',
            color=color, fontsize=10, ha='center', fontweight='bold')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title('Original Data\n+ Principal Component Directions', fontsize=10)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# Plot 2: data in PCA coordinates
ax2 = axes[1]
ax2.scatter(X_pca[:,0], X_pca[:,1], alpha=0.3, s=15, color='orange')
ax2.axhline(0,'gray',lw=0.5); ax2.axvline(0,'gray',lw=0.5)
ax2.set_xlabel(f'PC1 ({pct_var[0]:.1f}% variance)')
ax2.set_ylabel(f'PC2 ({pct_var[1]:.1f}% variance)')
ax2.set_title('After PCA Rotation\n(PC1 = horizontal = most variance)', fontsize=10)
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)

# Plot 3: scree plot
ax3 = axes[2]
bars = ax3.bar(['PC1', 'PC2'], pct_var, color=['red', 'orange'], edgecolor='black', linewidth=0.5)
ax3.set_ylabel('Variance Explained (%)')
ax3.set_title('Scree Plot: Variance per Component', fontsize=10)
ax3.set_ylim(0, 110)
for bar, pct in zip(bars, pct_var):
    ax3.text(bar.get_x() + bar.get_width()/2, pct + 1, f'{pct:.1f}%',
             ha='center', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle('PCA by Hand Using SVD', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 4: Visualize Column Space & Null Space

## The goal
For the matrix:
```
A = [[1, 2],
     [2, 1],
     [1, 1]]
```
- Visualize the **column space** (2D plane in ℝ³)
- Analyze the **null space** (only the zero vector, since A has full column rank)
- Show that Ax for many random x all land on the column space

In [ ]:
# ─────────────────────────────────────────────────────────
# COLUMN SPACE AND NULL SPACE
# ─────────────────────────────────────────────────────────

print("=== Part 4: Column Space and Null Space ===")
print()

A = np.array([[1., 2.],
              [2., 1.],
              [1., 1.]])   # shape (3, 2)

rank = np.linalg.matrix_rank(A)
nullity = A.shape[1] - rank

print(f"Matrix A (3×2):\n{A}")
print(f"\nShape: {A.shape}  →  m=3 (rows), n=2 (columns)")
print(f"Rank:    {rank}  = number of independent columns")
print(f"Nullity: {nullity}  = n - rank = {A.shape[1]} - {rank}")
print(f"Rank + Nullity = {rank} + {nullity} = {rank+nullity} = n ✓")

print()
print("=== Column space ===")
print(f"Column 1 of A: {A[:,0]}")
print(f"Column 2 of A: {A[:,1]}")
print(f"Column space = all vectors a₁×{A[:,0]} + a₂×{A[:,1]} for any scalars a₁, a₂")
print(f"             = a 2D PLANE through the origin in ℝ³")
print()

print("=== Null space ===")
if nullity == 0:
    print(f"Nullity = 0 → null space = {{0}} only")
    print(f"Only the zero vector maps to zero: A @ [0,0]ᵀ = {A @ np.array([0.,0.])}")
    print(f"Any non-zero x gives a non-zero Ax!")
    # Verify: try some vectors
    test_x = np.array([1., -1.])
    print(f"A @ [1,-1]ᵀ = {A @ test_x}  (non-zero ✓)")
else:
    U, s, Vt = np.linalg.svd(A, full_matrices=True)
    null_space = Vt[rank:, :]
    print(f"Null space basis vectors:")
    for i, nv in enumerate(null_space):
        print(f"  n{i+1} = {nv.round(4)}")
        print(f"  A @ n{i+1} = {(A @ nv).round(8)}  (should be zero)")

In [ ]:
# ── 3D Visualization of the column space ─────────────────
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 5))

# Left: show the column space plane
ax1 = fig.add_subplot(121, projection='3d')
col1 = A[:, 0]; col2 = A[:, 1]

# Draw the plane
c1_range = np.linspace(-2, 2, 15)
c2_range = np.linspace(-2, 2, 15)
C1, C2 = np.meshgrid(c1_range, c2_range)
plane_pts = C1[:,:,None]*col1 + C2[:,:,None]*col2

ax1.plot_surface(plane_pts[:,:,0], plane_pts[:,:,1], plane_pts[:,:,2],
                 alpha=0.25, color='steelblue')

# Draw the two column vectors
for vec, color, label in [(col1,'red','col1=[1,2,1]'), (col2,'orange','col2=[2,1,1]')]:
    ax1.quiver(0,0,0, vec[0],vec[1],vec[2], color=color, arrow_length_ratio=0.1, lw=2)
    ax1.text(vec[0]+0.1, vec[1]+0.1, vec[2]+0.1, label, color=color, fontsize=9)

ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
ax1.set_title('Column Space = Plane in ℝ³\nspanned by 2 column vectors', fontsize=10)
ax1.set_xlim(-3,4); ax1.set_ylim(-3,4); ax1.set_zlim(-3,4)

# Right: show that all Ax outputs land on the plane
ax2 = fig.add_subplot(122, projection='3d')
np.random.seed(0)
x_inputs = np.random.randn(50, 2)    # 50 random 2D input vectors
images = (A @ x_inputs.T).T          # apply A to each → all 3D vectors

ax2.scatter(images[:,0], images[:,1], images[:,2], color='steelblue', s=20, alpha=0.7)
ax2.plot_surface(plane_pts[:,:,0], plane_pts[:,:,1], plane_pts[:,:,2],
                 alpha=0.15, color='steelblue')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')
ax2.set_title('50 random Ax outputs\n(ALL land on the column space plane!)', fontsize=10)
ax2.set_xlim(-6,6); ax2.set_ylim(-6,6); ax2.set_zlim(-6,6)

plt.suptitle('Column Space: The Set of All Possible Outputs of Ax', fontsize=12)
plt.tight_layout()
plt.show()
print("Every output of A lives on a 2D plane — because A has rank 2!")

---
# Part 5: numpy.linalg.eig vs scipy.linalg.eig

## The goal
Understand when to use each function, what each returns, and which is more appropriate for different use cases.

In [ ]:
# ─────────────────────────────────────────────────────────
# COMPARISON: numpy.linalg.eig vs scipy.linalg.eig
# ─────────────────────────────────────────────────────────

print("=== Part 5: numpy.linalg.eig vs scipy.linalg.eig ===")
print()

# TEST 1: Symmetric matrix
np.random.seed(42)
M = np.random.randn(4, 4)
A_sym = M + M.T   # make symmetric

print("Test 1: Symmetric matrix")
print(f"A_sym =\n{A_sym.round(3)}")
print()

# numpy general
val_np, vec_np   = np.linalg.eig(A_sym)
# numpy symmetric-optimized
val_nph, vec_nph = np.linalg.eigh(A_sym)   # 'h' = Hermitian = symmetric for real matrices
# scipy general
val_sp, vec_sp   = scipy_linalg.eig(A_sym)
# scipy symmetric-optimized
val_sph, vec_sph = scipy_linalg.eigh(A_sym)

print(f"np.linalg.eig   : {np.sort(val_np.real).round(4)}  (may have tiny imaginary parts)")
print(f"np.linalg.eigh  : {np.sort(val_nph).round(4)}  (sorted, guaranteed real)")
print(f"scipy.linalg.eig: {np.sort(val_sp.real).round(4)}")
print(f"scipy.linalg.eigh:{np.sort(val_sph).round(4)}")
print(f"All agree: {np.allclose(np.sort(val_np.real), np.sort(val_nph))}")

In [ ]:
# TEST 2: Non-symmetric (can have complex eigenvalues)
print("\n=== Test 2: Non-symmetric matrix (rotation) ===")
theta = np.pi / 3   # 60 degree rotation
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

val_R_np, _ = np.linalg.eig(R)
print(f"Rotation matrix eigenvalues: {val_R_np}")
print(f"These are COMPLEX numbers! (a+bj = rotation)")
print(f"np.linalg.eigh would FAIL on non-symmetric matrices")
print(f"→ Always check if your matrix is symmetric before choosing eig vs eigh")

print()
# TEST 3: Scipy can also solve generalized eigenvalue problems: Av = λBv
print("=== Test 3: Generalized eigenvalue problem (scipy only) ===")
print("Find λ such that A @ v = λ × B @ v")
print("(numpy cannot do this, scipy can)")

A3 = np.array([[4., 2.], [2., 3.]])
B3 = np.array([[2., 0.], [0., 1.]])
val_gen, vec_gen = scipy_linalg.eig(A3, b=B3)
print(f"A = {A3.tolist()}, B = {B3.tolist()}")
print(f"Generalized eigenvalues: {val_gen.real.round(4)}")
print("(numpy.linalg.eig has no 'b=' argument — scipy needed here)")

In [ ]:
# TEST 4: Speed comparison
print("\n=== Test 4: Speed comparison ===")
n_size = 300
A_large = np.random.randn(n_size, n_size)
A_large = A_large + A_large.T  # make symmetric

functions = [
    ("np.linalg.eig",   lambda: np.linalg.eig(A_large)),
    ("np.linalg.eigh",  lambda: np.linalg.eigh(A_large)),
    ("scipy.linalg.eig",  lambda: scipy_linalg.eig(A_large)),
    ("scipy.linalg.eigh", lambda: scipy_linalg.eigh(A_large)),
]

for name, fn in functions:
    start = time.time()
    for _ in range(10): fn()
    elapsed = (time.time() - start) / 10
    print(f"  {name:<25}: {elapsed*1000:.1f} ms")

print()
print("Summary: eigh (symmetric-specific) is faster and more stable than eig")

In [ ]:
# ── Final comparison table ───────────────────────────────
print("\n" + "="*70)
print("WHEN TO USE WHICH FUNCTION")
print("="*70)
print()
comparison = [
    ["Function",             "Matrix type",     "Eigenvalues",     "Extras",             "When to use"],
    ["-"*18,                  "-"*15,             "-"*15,             "-"*18,               "-"*25],
    ["np.linalg.eig",        "Any square",      "May be complex",  "—",                  "General non-symmetric"],
    ["np.linalg.eigh",       "Symmetric only",  "Always real",     "Sorted ascending",   "BEST for symmetric ✓"],
    ["scipy.linalg.eig",     "Any square",      "May be complex",  "Left vectors, gen.", "Generalized Av=λBv"],
    ["scipy.linalg.eigh",    "Symmetric only",  "Always real",     "Sorted, generalized","Sym. generalized Av=λBv"],
]
for row in comparison:
    print(f"  {row[0]:<22} {row[1]:<18} {row[2]:<18} {row[3]:<22} {row[4]}")

print()
print("GOLDEN RULE:")
print("  If your matrix is symmetric (A=Aᵀ) → use eigh (faster, more stable, sorted)")
print("  If your matrix is general          → use eig")
print("  If you need Av = λBv              → use scipy.linalg.eig(A, b=B)")

---
# Lab Summary

## What you built

| Part | What you built | Key insight |
|---|---|---|
| 1 | Matrix multiply with 3 nested loops | Every element = one dot product; NumPy is 100-1000x faster |
| 2 | Eigendecomposition + verification | A = VΛVᵀ; matrix powers trivial via eigenvalues |
| 3 | PCA by hand (5 steps) | Center → SVD → V gives directions, s²/(n-1) gives variance |
| 4 | Column/null space visualization | All Ax live on the column space plane; rank + nullity = n |
| 5 | eig vs eigh comparison | For symmetric: always use eigh (faster, guaranteed real, sorted) |

---

## The big picture

Every ML model, at its core, is:
- A dataset X stored as a **matrix**
- Predictions computed via **matrix multiplication** Z = XW
- Weights found by **solving a linear system** (XᵀX)w = Xᵀy
- Directions of information found via **eigendecomposition / SVD**
- Model quality measured via **norms** ‖y - ŷ‖₂

**You now understand the mathematical foundation of machine learning.** 🎉

---

## What's next?
- **Week 2**: Calculus for ML (gradients, backpropagation)
- **Week 3**: Probability & Statistics for ML
- **Week 4**: Putting it all together — implementing neural networks from scratch